# CRIVO — O pensador: rede recursiva pequena que raciocina sobre o quadro

**Ideia:** no CRIVO, o Transformer 17M é a rede da linguagem (fala). Pensar é outra função. Este experimento treina **do zero** uma rede recursiva pequena (~0,4M parâmetros: 2 camadas aplicadas várias vezes sobre o próprio estado, ideia dos modelos recursivos pequenos como o TRM) que recebe o problema **já estruturado** — premissas, regras "se… então…", negações e a pergunta — e decide:

- **status:** sustentado, refutado, indeterminado ou conflito;
- **prova:** quais premissas bastam (o enumerador do CRIVO confere a prova proposta);
- **perguntar:** qual informação pedir quando falta.

**Professor:** `raciocinio_ativo.py` (o raciocínio lógico do próprio CRIVO). Nenhum modelo de terceiros.

**Conjuntos congelados** (conferidos por SHA): `blind_iid` (mesma distribuição do treino), `ood_profundidade` (provas de 4 a 6 regras; o treino só vê até 3) e `ood_tamanho` (11 a 16 átomos; o treino só vê até 8, e o runtime do CRIVO recusa mais de 10).

**Gates** (no número de voltas do treino): exato ≥ 95% no iid, ≥ 80% em profundidade e tamanho; afirmações erradas da rede ≤ 2%; zero afirmação errada com a política (prova conferida + base consistente); invariância a renomear/reordenar ≥ 98%. O resultado grava `ativo_no_chat: false` — passar nos gates não coloca nada no chat.

**Como rodar:** Ambiente de execução → GPU (T4) → Executar tudo. Leva cerca de 30–60 min. Tudo fica em `MyDrive/CRIVO/pensador-20261012/` (cada execução numa pasta nova).


In [ ]:
# Célula 0 — GPU e Drive
import os, sys, json, subprocess, hashlib, shutil
from pathlib import Path
from datetime import datetime
import torch
if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__, flush=True)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PASSOS = 20000        # passos de treino do pensador recursivo
VOLTAS = 8            # quantas vezes o bloco é aplicado (pensar mais = mais voltas)
RODAR_RASO = True     # controle: mesma rede com 1 volta só, para medir o ganho da recursão
MAX_HORAS = 2.5

RAIZ = Path("/content/drive/MyDrive/CRIVO/pensador-20261012")
SAIDA = RAIZ / datetime.now().strftime("execucao_%Y%m%d_%H%M%S")   # nunca sobrescreve
SAIDA.mkdir(parents=True, exist_ok=True)
print("Saída:", SAIDA)


In [ ]:
# Célula 1 — código e conjuntos congelados
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
EXP = REPO / "experimentos" / "pensador_20261012"
ESPERADO = "20deb41adcb704e791027ba5939feb253f6cbd59943de633389eee841500c3df"
obtido = hashlib.sha256((EXP / "conjuntos_pensador.json").read_bytes()).hexdigest()
assert obtido == ESPERADO, "conjuntos congelados diferentes do esperado: " + obtido
(SAIDA / "commit.txt").write_text(COMMIT + "\n")
print("commit", COMMIT, "| conjuntos conferidos", ESPERADO[:12])


In [ ]:
# Célula 2 — treino do pensador recursivo (escolha pelo dev; blind só no fim)
def rodar(destino, voltas):
    destino.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-u", str(EXP / "treinar_pensador.py"), "--saida", str(destino),
           "--passos", str(PASSOS), "--voltas", str(voltas), "--processos", "2", "--max-horas", str(MAX_HORAS)]
    with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for linha in p.stdout:
            print(linha, end="", flush=True)
    if p.returncode:
        raise SystemExit("treino falhou (código %d)" % p.returncode)
    return json.loads((destino / "resultado_pensador.json").read_text())

recursivo = rodar(SAIDA / "recursivo", VOLTAS)


In [ ]:
# Célula 3 — controle: a mesma rede com 1 volta (sem recursão)
raso = rodar(SAIDA / "raso", 1) if RODAR_RASO else None


In [ ]:
# Célula 4 — resumo
def linha(nome, r, v):
    m = r["avaliacao"][nome]["voltas_%d" % v]
    return "%-17s exato %5.1f%%  status %5.1f%%  erradas(rede) %4.1f%%  cobertura segura %5.1f%%" % (
        nome, 100 * m["exato"], 100 * m["status_acc"], 100 * m["afirmacoes_erradas_brutas"], 100 * m["politica_cobertura"])
for titulo, r, v in (("RECURSIVO (%d voltas)" % VOLTAS, recursivo, VOLTAS), ("RASO (1 volta)", raso, 1)):
    if r is None:
        continue
    print("\n" + titulo, "| parâmetros", r["parametros"], "| passo escolhido", r["passo_escolhido"])
    for nome in ("blind_iid", "ood_profundidade", "ood_tamanho"):
        print("  " + linha(nome, r, v))
    print("  invariância", round(100 * r["invariancia_status"], 1), "% | gates", r["gates"])
print("\nAPROVADO (recursivo):", recursivo["aprovado"], "| ativo_no_chat:", recursivo["ativo_no_chat"])
print("Arquivos em", SAIDA, "— mande um print deste resumo.")
